# ForTy: inspect publisher metadata

Run this short notebook on **CPU**. It collects the public dataset metadata
needed to investigate stored class IDs and downloads a small diagnostic ZIP.
It does not train models or read TFRecord shards.

Metadata may contain no class-name mapping. A successful download alone does
not establish the meaning of numeric labels. Keep label names unresolved until
authoritative evidence is found; never infer a replacement map from geography.


In [ ]:
import datetime
import hashlib
import importlib
import json
from pathlib import Path, PurePosixPath
import uuid
import zipfile

import tensorflow as tf

SOURCE = "gs://forest_typology/forty_v1/1.0.0"
MAX_FILE_BYTES = 2 * 1024 * 1024
MAX_TOTAL_BYTES = 8 * 1024 * 1024
MAX_FILES = 32
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = Path("/content/forty-source-checks") / (stamp + "-" + uuid.uuid4().hex[:8])
OUTPUT.mkdir(parents=True, exist_ok=False)
print("This attempt:", OUTPUT)


In [ ]:
def collect_metadata(source, output, gfile):
    report = {
        "schema_version": 1, "source": source,
        "mapping_status": "unresolved", "label_names": None,
        "files": [], "errors": [], "non_record_objects": [],
        "limits": {"per_file_bytes": MAX_FILE_BYTES,
                   "total_bytes": MAX_TOTAL_BYTES, "files": MAX_FILES},
    }
    retained = []
    try:
        names = sorted(gfile.listdir(source))
        report["non_record_objects"] = [name for name in names if ".tfrecord" not in name]
    except Exception as error:
        report["errors"].append({"stage": "list", "error": str(error)})
        return report, retained
    total = 0
    for name in report["non_record_objects"]:
        if PurePosixPath(name).name != name or not name.endswith((".json", ".txt", ".pbtxt")):
            continue
        remote = source + "/" + name
        entry = {"name": name, "source": remote}
        report["files"].append(entry)
        try:
            if gfile.isdir(remote):
                entry["status"] = "directory_skipped"
                continue
            size = int(gfile.stat(remote).length)
            entry["reported_bytes"] = size
            if size > MAX_FILE_BYTES or total + size > MAX_TOTAL_BYTES or len(retained) >= MAX_FILES:
                entry["status"] = "budget_skipped"
                continue
            with gfile.GFile(remote, "rb") as handle:
                payload = handle.read(MAX_FILE_BYTES + 1)
            if len(payload) > MAX_FILE_BYTES or total + len(payload) > MAX_TOTAL_BYTES:
                entry["status"] = "budget_skipped"
                continue
            # A prefix avoids collisions with our own inspection report names.
            local = output / ("publisher-" + name)
            local.write_bytes(payload)
            retained.append(local)
            total += len(payload)
            entry.update({"status": "saved", "archive_name": local.name,
                          "bytes": len(payload), "sha256": hashlib.sha256(payload).hexdigest()})
        except Exception as error:
            entry.update({"status": "failed", "error": str(error)})
    report["downloaded_bytes"] = total
    return report, retained


def inspect_tfds(source, report, output, loader):
    try:
        tfds = loader("tensorflow_datasets")
        builder = tfds.builder_from_directory(source)
        feature = builder.info.features["segmentation_labels"]
        names = getattr(feature, "names", None)
        report["tfds"] = {"version": tfds.__version__, "label_feature": str(feature),
                          "metadata": str(builder.info.metadata)[:100000]}
        if names is not None:
            report["label_names"] = list(names)[:1024]
            report["mapping_status"] = "names_present_requires_review"
        info_file = output / "tfds-info.txt"
        info_file.write_text(str(builder.info)[:100000], encoding="utf-8")
        return info_file
    except Exception as error:
        report["errors"].append({"stage": "tfds_info", "error": str(error)})
        return None


report, retained = collect_metadata(SOURCE, OUTPUT, tf.io.gfile)
info_file = inspect_tfds(SOURCE, report, OUTPUT, importlib.import_module)
if info_file is not None:
    retained.append(info_file)
report["scope"] = "Publisher metadata only. No class remapping, training or TFRecord reads."
report_file = OUTPUT / "inspection.json"
report_file.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
retained.append(report_file)
print(json.dumps(report, indent=2))


In [ ]:
ARCHIVE = OUTPUT.parent / ("forty-source-metadata-" + OUTPUT.name + ".zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in retained:
        archive.write(path, path.name)
if ARCHIVE.stat().st_size >= 32 * 1024 * 1024:
    raise RuntimeError("Report archive exceeds the 32 MiB transfer limit")
print("Saved:", ARCHIVE, "bytes:", ARCHIVE.stat().st_size)
print("Mapping status:", report["mapping_status"])
print("Download and retain this ZIP with the metadata report.")
from google.colab import files
files.download(str(ARCHIVE))
